# 수도권제1순환고속도로 속도 예측 — Google Colaboratory
>⚠️ Google Colaboratory 런타임 환경 기준으로 작성되어 있습니다.

## 런타임 설정

Colab(T4 GPU)과 CPU 전용 환경 모두에서 동일한 노트북을 사용할 수 있도록 환경 구성을 자동화했습니다.

In [ ]:
import importlib
import importlib.util
import os
import shutil
import subprocess
import sys
from pathlib import Path


def run_command(command, *, env=None):
    if isinstance(command, str):
        printable = command
    else:
        printable = ' '.join(str(c) for c in command)
    print(f'$ {printable}')
    subprocess.run(command, check=True, shell=isinstance(command, str), env=env)


try:
    IS_COLAB = importlib.util.find_spec('google.colab') is not None
except ModuleNotFoundError:
    IS_COLAB = False
PROJECT_ROOT = Path.cwd().resolve()

if IS_COLAB:
    apt_packages = ['htop', 'fonts-nanum']
    run_command('apt-get update')
    run_command('apt-get install -y ' + ' '.join(apt_packages))
else:
    print('ℹ️ Skipping apt-get because we are not running inside Google Colab.')

pip_packages: list[str] = []


def require(package: str, *, import_name: str | None = None, minimum: str | None = None, spec: str | None = None) -> None:
    name = import_name or package
    try:
        module = importlib.import_module(name)
        if minimum is not None:
            from packaging import version

            current = getattr(module, '__version__', None)
            if current is None or version.parse(current) < version.parse(minimum):
                raise ImportError
    except Exception:
        pkg_spec = spec or (f'{package}>={minimum}' if minimum else package)
        pip_packages.append(pkg_spec)


require('matplotlib', minimum='3.8')
require('polars', minimum='1.6.0')
require('xlsxwriter')
require('fastexcel')
require('openpyxl')
require('nbformat', minimum='5.10')
require('nbclient', minimum='0.10')
require('torch', minimum='2.6')
require('tensordict')

if pip_packages:
    run_command([sys.executable, '-m', 'pip', 'install', '--upgrade', '--prefer-binary', *pip_packages])
else:
    print('ℹ️ Required Python packages already installed.')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

try:
    import matplotlib.font_manager as fm
    import matplotlib.pyplot as plt
except ImportError:
    print('ℹ️ Matplotlib not available after installation; skipping font configuration.')
else:
    mpl_cache = Path.home() / '.cache' / 'matplotlib'
    if mpl_cache.exists():
        shutil.rmtree(mpl_cache)

    if any('NanumGothic' in f.name for f in fm.fontManager.ttflist):
        plt.rc('font', family='NanumGothic')
    else:
        print('ℹ️ NanumGothic font not found; using default Matplotlib font.')
    plt.rcParams['axes.unicode_minus'] = False

print('✅ Setup complete.')


In [ ]:
require('torchscale')
require('transformer-engine[pytorch]')
print('✅ Optional setup complete.')

In [ ]:
import torch

device_note = 'CUDA' if torch.cuda.is_available() else 'CPU/MPS'
print(f'✅ PyTorch {torch.__version__} imported successfully ({device_note} runtime).')


## 모델 및 PyTorch 로드

작업 디렉터리와 RAW 데이터 위치를 환경 변수로 조정할 수 있습니다.

In [ ]:
import os

if IS_COLAB:
    from google.colab import drive

    drive.mount('/content/drive', force_remount=False)
    print('🔁 Google Drive를 마운트했습니다.')

os.chdir(os.path.join(os.getcwd(), "drive/My Drive/Colab Notebooks/수도권제1순환고속도로 통행 속도 추세 분석하기 (vNext)/"))

import torch
import stnet
from stnet.utils.capability import get_device

device = get_device()
print(f'✅ Torch initialized with {device.type.upper()} (PyTorch {torch.__version__}).')


## RAW 데이터 로드

모든 시트를 Polars LazyFrame으로 적재하여 후속 전처리에 활용합니다.

In [ ]:
import os
import polars as pl
import openpyxl

xlsx_path = os.path.join(os.getcwd(), "수도권제1순환고속도로.xlsx")
workbook = openpyxl.load_workbook(xlsx_path, read_only=True, data_only=True)
sheet_names = workbook.sheetnames

def _read_lazy(sheet_name: str) -> pl.LazyFrame:
    return pl.read_excel(xlsx_path, sheet_name=sheet_name).lazy()

raw_data = {sheet_name: _read_lazy(sheet_name) for sheet_name in sheet_names}
workbook.close()
print(f'✅ {len(sheet_names)}개 시트를 로드했습니다: {xlsx_path}')

## 피처, 라벨 인코딩

In [ ]:
import re
from typing import List, Sequence, Tuple, Union

import numpy as np

def _is_hour_col(name: str) -> bool:
    s = str(name)
    return len(s) == 3 and s.endswith('시') and s[:2].isdigit()

def schema_names(df: Union[pl.DataFrame, pl.LazyFrame]) -> List[str]:
    if isinstance(df, pl.DataFrame):
        return list(df.columns)
    if isinstance(df, pl.LazyFrame):
        if hasattr(df, 'schema'):
            schema = df.schema
            if hasattr(schema, 'names'):
                return list(schema.names())
            if isinstance(schema, dict):
                return list(schema.keys())
        if hasattr(df, 'collect_schema'):
            return list(df.collect_schema().names())
    raise TypeError('df must be a Polars DataFrame or LazyFrame.')

def parse(name: str) -> Tuple[int, str]:
    m = re.search(r'^\d+', name)
    month = int(m.group()) if m else None
    daytype = '평일' if '평일' in name else '주말·공휴일'
    return month, daytype

def cleanse(
    df: Union[pl.DataFrame, pl.LazyFrame],
    month: int,
    daytype: str,
) -> pl.DataFrame:
    hour_cols = [c for c in HOURS if c in schema_names(df)]
    lazy_df = df.lazy() if isinstance(df, pl.DataFrame) else df

    return (
        lazy_df
        .with_columns(
            pl.col('구간').cast(pl.String),
            pl.col('방향').cast(pl.String),
        )
        .filter(pl.col('구간').is_not_null() & pl.col('구간').str.contains('-', literal=True))
        .with_columns([pl.col(c).cast(pl.Float32, strict=False) for c in hour_cols])
        .with_columns(
            pl.col('구간').str.replace_all(r'\s+', '').alias('구간'),
            pl.col('방향').str.strip_chars().alias('방향'),
            pl.col('구간').str.split('-').list.sort().list.join('-').alias('구간ID'),
        )
        .unpivot(
            index=['구간', '구간ID', '방향'],
            on=hour_cols,
            variable_name='시간문자',
            value_name='속도',
        )
        .with_columns(
            pl.lit(month).alias('월'),
            pl.lit(DAY2ID[daytype]).alias('요일타입_id'),
            pl.when(pl.col('방향') == '상행').then(0)
              .when(pl.col('방향') == '하행').then(1)
              .otherwise(None)
              .cast(pl.Int8, strict=False)
              .alias('방향_id'),
            pl.col('시간문자').str.replace_all('시', '').cast(pl.Int32, strict=False).alias('시간'),
            pl.col('속도').cast(pl.Float32, strict=False),
        )
        .select(['월', '요일타입_id', '방향_id', '구간ID', '시간', '속도'])
        .filter(pl.col('방향_id').is_not_null())
        .group_by(['월', '요일타입_id', '방향_id', '구간ID', '시간'])
        .agg(pl.col('속도').mean().alias('속도'))
        .collect()
    )

def nanmean_grid(grid: np.ndarray) -> np.ndarray:
    with np.errstate(invalid='ignore'):
        col_mean = np.nanmean(grid, axis=0, keepdims=True)
    col_mean = np.where(np.isnan(col_mean), 0.0, col_mean)
    return np.where(np.isnan(grid), col_mean, grid)

In [ ]:
HOURS = [f'{h:02d}시' for h in range(24)]
DAY2ID = {'평일': 0, '주말·공휴일': 1}
DIR2ID = {'상행': 0, '하행': 1}
DIR_ID2NAME = {v: k for k, v in DIR2ID.items()}
long_parts: List[pl.DataFrame] = []

for name in sheet_names:
    month, daytype = parse(name)
    if month is None:
        raise ValueError(f'시트명에서 월 정보를 찾을 수 없습니다: {name}')
    long_parts.append(cleanse(raw_data[name], month, daytype))

df = pl.concat(long_parts, how='vertical', rechunk=True)
segments = df.select('구간ID').unique().sort('구간ID').get_column('구간ID').to_list()
seg2idx = {s: i for i, s in enumerate(segments)}
S, T = len(segments), 24
train_data_3d: dict[Tuple[int, int, int], torch.Tensor] = {}

for (m, d_id, dir_id), g in df.group_by(['월', '요일타입_id', '방향_id']):
    grid = np.full((T, S, 1), np.nan, dtype=np.float32)
    for (t, seg, v) in g.select(['시간', '구간ID', '속도']).iter_rows():
        s = seg2idx[seg]
        if v is not None and not np.isnan(v):
            grid[int(t), int(s), 0] = float(v)
    grid = nanmean_grid(grid)
    y = torch.from_numpy(grid)
    key = (int(m), int(d_id), int(dir_id))
    train_data_3d[key] = y

keys_sorted = sorted(train_data_3d.keys())
features_tensor = torch.tensor(keys_sorted, dtype=torch.float32)
label_tensor = torch.stack([train_data_3d[k] for k in keys_sorted], dim=0)
labels_flat = label_tensor.view(len(keys_sorted), -1)
label_shape = tuple(label_tensor.shape[1:])

print(f'✅ {len(train_data_3d)}개의 학습 샘플을 준비했습니다. (Segments={S}, Hours={T})')
print(f'Features tensor shape: {features_tensor.shape}, label tensor shape: {label_tensor.shape}')


## 모델 생성

`stnet.config.ModelConfig`와 `stnet.config.PatchConfig`는 `modeling_type` 인자를 통해 입력 격자의 도메인을 설정합니다.
런타임 모듈에서도 동일한 변수들을 재노출하므로 기존 코드를 그대로 사용할 수 있습니다.
다음 별칭은 모두 대소문자와 무관하게 지원됩니다.

- 공간 전용: `ss`, `spatial`
- 시간 전용: `tt`, `temporal`
- 시공간 결합: `st`, `ts`, `spatiotemporal`


In [ ]:
from stnet import PatchConfig, build_config

patch = PatchConfig(
    is_cube=True,
    grid_size_3d=(T, S, 1),
    patch_size_3d=(1, 1, 1),
    use_padding=True,
)

if device.type == 'cuda':
    config = build_config(
        device=device,
        microbatch=64,
        dropout=0.05,
        normalization_method='rmsnorm',
        depth=1152,
        heads=6,
        spatial_depth=4,
        temporal_depth=4,
        mlp_ratio=3.0,
        drop_path=0.05,
        spatial_latents=80,
        temporal_latents=60,
        modeling_type='spatiotemporal',
        patch=patch,
        enable_compilation=True,
        compile_mode='max-autotune',
    )
else:
    config = build_config(
        device=device,
        microbatch=32,
        dropout=0.10,
        normalization_method='layernorm',
        depth=512,
        heads=2,
        spatial_depth=2,
        temporal_depth=2,
        mlp_ratio=2.0,
        drop_path=0.0,
        spatial_latents=32,
        temporal_latents=32,
        modeling_type='spatiotemporal',
        patch=patch,
        enable_compilation=False,
        compile_mode='default',
    )

print(config)


In [ ]:
in_dim = 3
out_shape = (T, S, 1)
model = stnet.new_model(config=config, in_dim=in_dim, out_shape=out_shape)
print(model)


##모델 학습

In [ ]:
num_samples = features_tensor.shape[0]
base_params = {
    'epochs': 30 if device.type == 'cuda' else 40,
    'batch_size': max(4, min(16, num_samples)),
    'base_lr': 3e-3,
    'weight_decay': 1e-4,
}
print(base_params)


In [ ]:
from stnet import learn as workflow_learn

train_dataset = {"X": features_tensor, "Y": label_tensor}
model = workflow_learn(
    model,
    train_dataset,
    epochs=int(base_params["epochs"]),
    batch_size=int(base_params["batch_size"]),
    base_lr=float(base_params["base_lr"]),
    weight_decay=float(base_params["weight_decay"]),
    val_frac=0.1,
)


##추론 데이터를 저장할 템플릿 생성

In [ ]:
import copy

template = copy.deepcopy(raw_data)
time_columns = [col for col in schema_names(next(iter(template.values()))) if _is_hour_col(col)]
template = {
    sheet_name: lf.with_columns([pl.lit(0).alias(col) for col in time_columns])
    for sheet_name, lf in template.items()
}

## 모델 추론 및 평가


In [ ]:
from stnet import infer as workflow_infer

infer_samples = {key: None for key in keys_sorted}
raw_predictions = workflow_infer(
    model,
    infer_samples,
    batch_size=int(base_params["batch_size"]),
)
pred_tensor = torch.stack([raw_predictions[key] for key in keys_sorted]).detach().cpu()
target_tensor = label_tensor.detach().cpu()
mae = torch.mean(torch.abs(pred_tensor - target_tensor)).item()
rmse = torch.sqrt(torch.mean((pred_tensor - target_tensor) ** 2)).item()
print(f'✅ Evaluation complete — MAE: {mae:.3f}, RMSE: {rmse:.3f}')

pred_dict = {keys_sorted[i]: pred_tensor[i] for i in range(len(keys_sorted))}


## 추론 데이터 저장

In [ ]:
from typing import Dict, Sequence, Tuple

def get_schema(raw_data: Dict[str, pl.LazyFrame]) -> Tuple[Sequence[str], Sequence[str]]:
    first_sheet = next(iter(raw_data.values()))
    cols = schema_names(first_sheet)
    idx_first_hour = min([i for i, c in enumerate(cols) if _is_hour_col(c)], default=len(cols))
    schema_left = list(cols[:idx_first_hour]) if idx_first_hour > 0 else ['구간', '방향']
    schema_right = HOURS
    return (schema_left, schema_right)

def categorize_poi(seg_id: str, dir_id: int, delim: str = '-') -> str:
    a, b = seg_id.split(delim, 1)
    return f'{a}{delim}{b}' if int(dir_id) == 0 else f'{b}{delim}{a}'

def to_long_polars(key: Tuple[int, int, int], grid: torch.Tensor, seg_ids: Sequence[str]) -> pl.DataFrame:
    m, day_id, dir_id = map(int, key)
    T_dim, S_dim, C_dim = grid.shape
    assert C_dim == 1
    g = grid.detach().to('cpu').float().numpy()
    recs = [
        (m, day_id, dir_id, seg_ids[s], t, float(g[t, s, 0]))
        for t in range(T_dim)
        for s in range(S_dim)
    ]
    return pl.DataFrame(recs, schema=['월', '요일타입_id', '방향_id', '구간ID', '시간', '속도'], orient='row')

def to_wide_polars(long_df: pl.DataFrame, index_cols: Sequence[str]) -> pl.DataFrame:
    w = long_df.pivot(values='속도', index=index_cols, columns='시간', aggregate_function='mean')
    time_cols = [c for c in w.columns if c not in index_cols]
    have = {int(c) for c in time_cols if str(c).isdigit()}
    for h in range(24):
        if h not in have:
            w = w.with_columns(pl.lit(None).alias(str(h)))
    w = w.select(list(index_cols) + [str(h) for h in range(24)])
    w = w.rename({str(h): f'{h:02d}시' for h in range(24)})
    return w.with_columns([
        pl.col('방향_id').cast(pl.Int32, strict=False),
        pl.col('구간ID').cast(pl.String),
    ])

def to_polars(
    template_lf: pl.LazyFrame,
    schema_left: Sequence[str],
    schema_right: Sequence[str],
    pred_long_for_sheet: pl.DataFrame,
) -> pl.DataFrame:
    temp = template_lf.collect()
    if '구간' not in temp.columns or '방향' not in temp.columns:
        raise ValueError("템플릿 시트에 '구간' 또는 '방향' 컬럼이 없습니다.")

    temp = temp.with_columns([
        pl.col('구간').cast(pl.String).str.split('-').list.sort().list.join('-').alias('구간ID'),
        pl.col('방향').map_elements(lambda s: DIR2ID.get(s, 1), return_dtype=pl.Int32).alias('방향_id'),
    ])

    wide_pred = to_wide_polars(pred_long_for_sheet, index_cols=['구간ID', '방향_id'])
    time_cols_in_template = [c for c in temp.columns if _is_hour_col(c)]
    joined = temp.join(wide_pred, on=['구간ID', '방향_id'], how='left', suffix='_pred')
    final = joined
    for h in schema_right:
        pred_col = f'{h}_pred'
        if pred_col in final.columns:
            final = final.with_columns(
                pl.when(pl.col(pred_col).is_not_null())
                  .then(pl.col(pred_col))
                  .otherwise(pl.col(h) if h in time_cols_in_template else pl.lit(None))
                  .alias(h)
            )
        elif h not in final.columns:
            final = final.with_columns(pl.lit(None).alias(h))
    final = final.drop([c for c in final.columns if c.endswith('_pred')] + ['구간ID', '방향_id'])
    keep_left = [c for c in schema_left if c in final.columns]
    final = final.select(keep_left + [c for c in schema_right if c in final.columns])
    cols = [h for h in schema_right if h in final.columns]
    final = final.with_columns([pl.col(h).round(0).cast(pl.UInt8, strict=False).alias(h) for h in cols])
    return final


In [ ]:
pred_longs = []
keys_sorted = sorted(pred_dict.keys())
for key in keys_sorted:
    grid = pred_dict[key]
    pred_longs.append(to_long_polars(key, grid, segments))
pred_long_df = pl.concat(pred_longs, how='vertical', rechunk=True)

schema_left, schema_right = get_schema(template)

pred_sheets: Dict[str, pl.DataFrame] = {}
for sheet_name, lf in template.items():
    month = int(re.search(r'^\d+', sheet_name).group())
    daytype = '평일' if '평일' in sheet_name else '주말·공휴일'
    d_id = DAY2ID[daytype]
    long_for_sheet = pred_long_df.filter((pl.col('월') == month) & (pl.col('요일타입_id') == d_id))
    final_df = to_polars(lf, schema_left=schema_left, schema_right=schema_right, pred_long_for_sheet=long_for_sheet)
    pred_sheets[sheet_name] = final_df

In [ ]:
import xlsxwriter
import os

pred_xlsx = os.path.join(os.getcwd(), "예측 데이터.xlsx")
workbook = xlsxwriter.Workbook(str(pred_xlsx))

for sheet_name, frame in pred_sheets.items():
    frame.write_excel(workbook=workbook, worksheet=sheet_name, autofit=True)

workbook.close()
print(f'✅ 예측 결과를 저장했습니다: {pred_xlsx}')